<a href="https://colab.research.google.com/github/ealmeida04/logica-programacao/blob/main/agregacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, year, col
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:

# Criando a sessão Spark
spark = SparkSession.builder.appName("YouTubeDataPreparation").getOrCreate()


Leia o arquivo 'videos-preparados.snappy.parquet' no dataframe 'df_video'

In [12]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  uploaded_file_name = fn

print(f"The uploaded file name is: {uploaded_file_name}")

Saving videos-preparados.snappy.parquet to videos-preparados.snappy.parquet
User uploaded file "videos-preparados.snappy.parquet" with length 245808 bytes
The uploaded file name is: videos-preparados.snappy.parquet


In [13]:
df_video = spark.read.parquet(uploaded_file_name)
df_video.show(5)
df_video.printSchema()

+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|202


Calcule a quantidade de registros para cada valor exclusivo da coluna "Palavra-chave"

In [14]:
# Calculate the count of records for each unique value in the 'Keyword' column
keyword_counts = df_video.groupBy("Keyword").count()

# Order the results by count in descending order for easier analysis
keyword_counts.orderBy(col("count").desc()).show(truncate=False)

+----------------+-----+
|Keyword         |count|
+----------------+-----+
|cnn             |50   |
|interview       |50   |
|crypto          |50   |
|data science    |50   |
|trolling        |50   |
|tutorial        |50   |
|marvel          |50   |
|game development|50   |
|mrbeast         |50   |
|physics         |50   |
|sat             |49   |
|history         |49   |
|cubes           |49   |
|reaction        |49   |
|sports          |49   |
|asmr            |49   |
|computer science|48   |
|food            |48   |
|how-to          |48   |
|machine learning|48   |
+----------------+-----+
only showing top 20 rows


Calcule a mídia da coluna "Interação" para cada valor único da coluna 'Palavra-chave'

In [15]:
from pyspark.sql.functions import avg

# Calculate the average of 'Interaction' for each unique 'Keyword'
keyword_interaction_avg = df_video.groupBy("Keyword").agg(avg("Interaction").alias("Average_Interaction"))

# Show the results, ordered by average interaction in descending order
keyword_interaction_avg.orderBy(col("Average_Interaction").desc()).show(truncate=False)

+-------------+--------------------+
|Keyword      |Average_Interaction |
+-------------+--------------------+
|animals      |9.55066085263158E7  |
|mrbeast      |6.896586282E7       |
|bed          |5.438209175E7       |
|music        |2.9691370304347824E7|
|history      |1.565269257142857E7 |
|cubes        |1.5043961224489795E7|
|mukbang      |1.1053630377777778E7|
|apple        |1.0873628214285715E7|
|sports       |8695551.632653061   |
|how-to       |7975134.5           |
|business     |7310180.020833333   |
|tutorial     |6936688.3           |
|marvel       |6834159.44          |
|food         |5352944.104166667   |
|movies       |4897436.318181818   |
|biology      |4192382.063829787   |
|lofi         |4167085.875         |
|physics      |3795529.38          |
|mathchemistry|3427342.7333333334  |
|interview    |3044867.04          |
+-------------+--------------------+
only showing top 20 rows


Calcule o valor máximo da coluna "Interaction" para cada valor único da coluna 'Keyword' e nomeie de 'Rank Interactions', em seguida ordena pela nova coluna em ordem decrescente

In [16]:
from pyspark.sql.functions import max, col

# Calculate the maximum of 'Interaction' for each unique 'Keyword'
keyword_max_interaction = df_video.groupBy("Keyword").agg(max("Interaction").alias("Rank Interactions"))

# Show the results, ordered by 'Rank Interactions' in descending order
keyword_max_interaction.orderBy(col("Rank Interactions").desc()).show(truncate=False)

+--------+-----------------+
|Keyword |Rank Interactions|
+--------+-----------------+
|animals |1593623628       |
|music   |922551152        |
|bed     |532691631        |
|history |440187490        |
|apple   |429916936        |
|mrbeast |300397699        |
|google  |239385460        |
|business|210025196        |
|cubes   |170925917        |
|sports  |106924567        |
|mukbang |87433858         |
|lofi    |86445177         |
|tutorial|69616442         |
|movies  |65253870         |
|marvel  |56247330         |
|how-to  |53053975         |
|food    |48754479         |
|physics |43463298         |
|asmr    |34411125         |
|nintendo|32268486         |
+--------+-----------------+
only showing top 20 rows


Calcule a média e a variância da coluna 'Visualizações' para cada valor único da coluna 'Palavra-chave'

In [17]:
from pyspark.sql.functions import avg, variance, col

# Calculate the average and variance of 'Views' for each unique 'Keyword'
keyword_views_stats = df_video.groupBy("Keyword").agg(
    avg("Views").alias("Average_Views"),
    variance("Views").alias("Variance_Views")
)

# Show the results, ordered by average views in descending order
keyword_views_stats.orderBy(col("Average_Views").desc()).show(truncate=False)

+-------------+--------------------+----------------------+
|Keyword      |Average_Views       |Variance_Views        |
+-------------+--------------------+----------------------+
|animals      |9.472396092105263E7 |8.3537868257472992E16 |
|mrbeast      |6.676400398E7       |3.8241236796058515E15 |
|google       |6.143966791111111E7 |1.02697972988153936E17|
|bed          |5.389322861363637E7 |1.1661048318545686E16 |
|music        |2.9364893260869566E7|1.9247971071879404E16 |
|history      |1.5353155530612245E7|4.253204661918686E15  |
|cubes        |1.4735344122448979E7|8.511756571903261E14  |
|mukbang      |1.0904772355555555E7|5.5860732389731794E14 |
|apple        |1.0746930452380951E7|4.299927977442589E15  |
|sports       |8601204.734693877   |3.097712025588381E14  |
|how-to       |7809284.916666667   |1.7758734965359956E14 |
|business     |7236354.520833333   |9.546545645303889E14  |
|tutorial     |6761032.02          |1.3696265968644572E14 |
|marvel       |6614079.56          |1.44

Calcule a média, o valor mínimo e o valor máximo de 'Visualizações' para cada valor único da coluna 'Palavra-chave', sem casas decimais

In [18]:
from pyspark.sql.functions import avg, min, max, col, floor
from pyspark.sql.types import IntegerType

# Calculate the average, min, and max of 'Views' for each unique 'Keyword'
keyword_views_summary = df_video.groupBy("Keyword").agg(
    floor(avg("Views")).cast(IntegerType()).alias("Average_Views"),
    min("Views").alias("Min_Views"),
    max("Views").alias("Max_Views")
)

# Show the results, ordered by average views in descending order
keyword_views_summary.orderBy(col("Average_Views").desc()).show(truncate=False)

+-------------+-------------+---------+----------+
|Keyword      |Average_Views|Min_Views|Max_Views |
+-------------+-------------+---------+----------+
|animals      |94723960     |23448    |1582262997|
|mrbeast      |66764003     |889300   |285526909 |
|google       |61439667     |8064     |2147483647|
|bed          |53893228     |4454     |524709805 |
|music        |29364893     |2944     |915457091 |
|history      |15353155     |6640     |434352213 |
|cubes        |14735344     |10146    |168546247 |
|mukbang      |10904772     |3618     |86169225  |
|apple        |10746930     |1954     |425478119 |
|sports       |8601204      |867      |106014469 |
|how-to       |7809284      |3311     |52061447  |
|business     |7236354      |2270     |208293677 |
|tutorial     |6761032      |19323    |68512549  |
|marvel       |6614079      |2813     |54583132  |
|food         |5252406      |47430    |48018833  |
|movies       |4862426      |2758     |65067408  |
|biology      |4121605      |55


Mostre o primeiro e o último 'Published At' para cada valor exclusivo da coluna 'Keyword'

In [19]:
from pyspark.sql.functions import min, max

# Calculate the first and last 'Published At' date for each unique 'Keyword'
keyword_published_at_range = df_video.groupBy("Keyword").agg(
    min("Published At").alias("First_Published_At"),
    max("Published At").alias("Last_Published_At")
)

# Show the results
keyword_published_at_range.orderBy("Keyword").show(truncate=False)

+----------------+------------------+-----------------+
|Keyword         |First_Published_At|Last_Published_At|
+----------------+------------------+-----------------+
|animals         |2009-07-03        |2022-08-24       |
|apple           |2016-11-02        |2022-08-24       |
|asmr            |2020-10-15        |2022-08-24       |
|bed             |2007-07-16        |2022-08-24       |
|biology         |2009-02-16        |2022-07-30       |
|business        |2009-10-25        |2022-08-24       |
|chess           |2019-09-12        |2022-08-24       |
|cnn             |2022-07-14        |2022-08-24       |
|computer science|2009-08-20        |2022-08-12       |
|crypto          |2022-03-11        |2022-08-24       |
|cubes           |2009-02-24        |2022-08-24       |
|data science    |2018-06-23        |2022-08-24       |
|education       |2008-07-25        |2022-08-24       |
|finance         |2012-11-27        |2022-08-24       |
|food            |2017-05-31        |2022-08-24 

Conte todos os 'títulos' de forma normal e todos os únicos e verifique se há diferença

In [20]:
# Count total number of titles
total_titles = df_video.count()
print(f"Total number of titles: {total_titles}")

# Count unique titles
unique_titles = df_video.select("Title").distinct().count()
print(f"Number of unique titles: {unique_titles}")

# Check for difference
if total_titles == unique_titles:
    print("There is no difference between the total number of titles and unique titles.")
else:
    difference = total_titles - unique_titles
    print(f"There is a difference. Total titles: {total_titles}, Unique titles: {unique_titles}. Difference: {difference}")

Total number of titles: 1869
Number of unique titles: 1854
There is a difference. Total titles: 1869, Unique titles: 1854. Difference: 15


Mostrar a quantidade de registros ordenados por ano em ordem ascendente

In [21]:
from pyspark.sql.functions import col

# Calculate the count of records for each unique 'Year'
year_counts = df_video.groupBy("Year").count()

# Order the results by 'Year' in ascending order
year_counts.orderBy(col("Year").asc()).show(truncate=False)

+----+-----+
|Year|count|
+----+-----+
|2007|2    |
|2008|1    |
|2009|9    |
|2010|6    |
|2011|4    |
|2012|12   |
|2013|6    |
|2014|10   |
|2015|15   |
|2016|34   |
|2017|47   |
|2018|57   |
|2019|86   |
|2020|158  |
|2021|229  |
|2022|1193 |
+----+-----+




Mostrar a quantidade de registros ordenados por ano e mês em ordem ascendente

In [22]:
from pyspark.sql.functions import col

# Calculate the count of records for each unique combination of 'Year' and 'Month'
year_month_counts = df_video.groupBy("Year", "Month").count()

# Order the results by 'Year' and then by 'Month' in ascending order
year_month_counts.orderBy(col("Year").asc(), col("Month").asc()).show(truncate=False)

+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2007|7    |1    |
|2007|12   |1    |
|2008|7    |1    |
|2009|2    |2    |
|2009|6    |2    |
|2009|7    |1    |
|2009|8    |1    |
|2009|10   |1    |
|2009|12   |2    |
|2010|3    |1    |
|2010|5    |2    |
|2010|6    |1    |
|2010|9    |1    |
|2010|10   |1    |
|2011|2    |1    |
|2011|5    |1    |
|2011|9    |1    |
|2011|10   |1    |
|2012|1    |1    |
|2012|2    |3    |
+----+-----+-----+
only showing top 20 rows


Calcule a média acumulativa de 'Likes' para cada 'Keyword' ao longo dos anos.

In [23]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col

In [24]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col

window_spec = Window.partitionBy("Keyword").orderBy("Year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
print("Window specification 'window_spec' defined.")

Window specification 'window_spec' defined.


In [25]:
from pyspark.sql.functions import avg

cumulative_avg_likes = df_video.withColumn(
    "Cumulative_Average_Likes",
    avg(col("Likes")).over(window_spec)
)

# Show the results, ordered by Keyword and Year
cumulative_avg_likes.orderBy("Keyword", "Year").show(truncate=False)

+--------------------------------------------------------------------------+-----------+------------+-------+--------+--------+----------+-----------+----+-----+-------------+--------------------+---------------------------------------------------------------+-------------------------------------------+------------------------+
|Title                                                                     |Video ID   |Published At|Keyword|Likes   |Comments|Views     |Interaction|Year|Month|Keyword Index|Features PCA        |Features Normal                                                |Features                                   |Cumulative_Average_Likes|
+--------------------------------------------------------------------------+-----------+------------+-------+--------+--------+----------+-----------+----+-----+-------------+--------------------+---------------------------------------------------------------+-------------------------------------------+------------------------+
|The Anima